# SPEI time-scale selection for Thrace rainfed drought — full analysis
### Multi-criterion ranking · dose–response · lag · yield validation · robustness

Given the district Landsat VHI (α = 0.5) and the station SPI/SPEI (1965–2024), this notebook
runs the manuscript's scale-selection analysis end-to-end and identifies the accumulation window
that best represents agricultural drought in Turkish Thrace.

**Inputs:** `trakya_district_vhi_vci_tci_monthly.csv`, `trakya_spi_spei_1965_2024.csv`, `Yield.xls`.

**Result (reproduces the paper):** **SPEI-3** ranks first on the composite score (top on Pearson,
Spearman, mutual information, dose–response amplitude, and lag-0 suitability), with SPEI-6 close
behind; 12-month scales are weakest. Sunflower yield tracks growing-season SPEI-3/SPEI-6
(r ≈ 0.45–0.51); wheat is weakly linked (winter-moisture buffered). SPEI-3 is the best scale in
5 of 6 α × season robustness scenarios.


## 0 · Load VHI, SPI/SPEI, yield

In [ ]:
import pandas as pd, numpy as np, unicodedata
from scipy import stats
from sklearn.feature_selection import mutual_info_regression
import matplotlib.pyplot as plt

TR=str.maketrans('ıİşŞçÇğĞöÖüÜ','iissccggoouu')
def fold(s):
    s=str(s).translate(TR); return ''.join(c for c in unicodedata.normalize('NFKD',s) if not unicodedata.combining(c)).lower().strip()

vhi=pd.read_csv('trakya_district_vhi_vci_tci_monthly.csv')
vhi['VHI']=0.5*vhi['VCI']+0.5*vhi['TCI']; vhi['key']=vhi['name'].map(fold)
S=pd.read_csv('trakya_spi_spei_1965_2024.csv'); S['key']=S['name'].map(fold)
merged=vhi.merge(S.drop(columns='name'),on=['key','year','month'],how='inner')

SCALES=[1,3,6,9,12]; INDICES=[f'{f}{k}' for f in ['SPI','SPEI'] for k in SCALES]
WIN=[6,7,8,9,10]        # vegetation-sensitive window
def clean(x,y):
    ok=np.isfinite(x)&np.isfinite(y); return x[ok],y[ok]
print('Districts:', merged.name.nunique(), '| overlap', int(merged.year.min()),'-',int(merged.year.max()))

## 1 · Multi-criterion evaluation

Each candidate index is scored on five criteria over Jun–Oct: **Pearson** (linear strength),
**Spearman** (monotonic), **mutual information** (nonlinear/general association; a computable
stand-in for MIC), **dose–response amplitude** (VHI difference between top and bottom index
terciles), and **lag-0 correlation** (real-time suitability). Each criterion is ranked across the
ten indices; the composite is the rank sum (max = 50).

In [ ]:
d=merged[merged.month.isin(WIN)].copy()

def lag0_r(idx):
    x,y=clean(d['VHI'].values,d[idx].values); return stats.pearsonr(x,y)[0]

rows=[]
for idx in INDICES:
    x,y=clean(d['VHI'].values,d[idx].values)
    pear=stats.pearsonr(x,y)[0]; spear=stats.spearmanr(x,y)[0]
    mi=mutual_info_regression(y.reshape(-1,1),x,random_state=0,n_neighbors=5)[0]
    q1,q3=np.quantile(y,[1/3,2/3]); amp=x[y>=q3].mean()-x[y<=q1].mean()
    rows.append({'index':idx,'pearson':pear,'spearman':spear,'MI':mi,'dose_amp':amp,'r_lag0':pear})
crit=pd.DataFrame(rows)
for c in ['pearson','spearman','MI','dose_amp','r_lag0']:
    crit[c+'_rank']=crit[c].rank()
crit['composite']=crit[[c+'_rank' for c in ['pearson','spearman','MI','dose_amp','r_lag0']]].sum(axis=1)
crit=crit.sort_values('composite',ascending=False).reset_index(drop=True)
print(crit[['index','pearson','spearman','MI','dose_amp','composite']].round(3).to_string(index=False))
print(f"\nTop scale: {crit.iloc[0]['index']} (composite {crit.iloc[0].composite:.0f}/50)")
crit.to_csv('scale_selection_multicriteria.csv',index=False)

# figure: composite bar
fig,ax=plt.subplots(figsize=(9,3.5))
c2=crit.sort_values('composite'); col=['#c0392b' if 'SPEI' in i else '#2980b9' for i in c2['index']]
ax.barh(c2['index'],c2['composite'],color=col); ax.set_xlabel('Composite score (5 criteria, max 50)')
ax.set_title('SPEI/SPI scale ranking for VHI (Jun–Oct)'); plt.tight_layout(); plt.show()

## 2 · Dose–response

VHI conditioned on drought severity. Using extreme bins (index < −1 dry, > 1 wet), the
dry→wet VHI swing is largest for the short-to-medium scales — the index that most cleanly
separates stressed from healthy vegetation.

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4))
# (a) swing by scale
sw=[]
for k in SCALES:
    dry=d.loc[d[f'SPEI{k}']<-1,'VHI'].mean(); wet=d.loc[d[f'SPEI{k}']>1,'VHI'].mean()
    sw.append({'scale':k,'dry':dry,'wet':wet,'swing':wet-dry})
sw=pd.DataFrame(sw)
ax[0].bar(sw.scale.astype(str),sw.swing,color='#8e44ad')
ax[0].set_xlabel('SPEI scale (months)'); ax[0].set_ylabel('VHI swing (wet − dry)')
ax[0].set_title('Dose–response amplitude')
# (b) binned VHI vs SPEI-3
bins=[-3,-1.5,-1,-0.5,0,0.5,1,1.5,3]; d['bin']=pd.cut(d['SPEI3'],bins)
mv=d.groupby('bin')['VHI'].mean()
xpos=range(len(mv))
ax[1].plot(list(xpos),mv.values,'o-',color='#c0392b')
ax[1].set_xticks(list(xpos)); ax[1].set_xticklabels([str(b) for b in mv.index],rotation=45,ha='right',fontsize=7)
ax[1].set_ylabel('mean VHI'); ax[1].set_title('VHI vs SPEI-3 (binned)')
plt.tight_layout(); plt.show()
print(sw.round(1).to_string(index=False))

## 3 · Lag analysis

Cross-correlation of VHI(t) against index(t − lag). A peak at **lag 0** means the index reflects
vegetation stress in the same month — ideal for near-real-time monitoring.

In [ ]:
LAGS=range(-3,4)
fig,ax=plt.subplots(figsize=(9,4))
for idx,style in [('SPEI3','-o'),('SPEI6','-s'),('SPEI12','-^')]:
    prof=[]
    for L in LAGS:
        sub=merged.sort_values(['key','year','month']).copy()
        sub['lag']=sub.groupby('key')[idx].shift(L)
        s=sub[sub.month.isin(WIN)]; x,y=clean(s['VHI'].values,s['lag'].values)
        prof.append(stats.pearsonr(x,y)[0])
    ax.plot(list(LAGS),prof,style,label=idx)
ax.axvline(0,color='k',lw=.6,ls='--'); ax.set_xlabel('lag (months, index leads VHI)')
ax.set_ylabel('Pearson r'); ax.set_title('VHI–SPEI cross-correlation'); ax.legend()
plt.tight_layout(); plt.show()

## 4 · Yield validation

Detrended district yield vs growing-season SPEI (wheat sampled at June, sunflower at August).
Sunflower — a summer crop — tracks the growing-season water balance strongly; wheat, winter-sown
and buffered by stored soil moisture, is weakly linked.

In [ ]:
def detrend(sr):
    x=sr.index.values.astype(float); y=sr.values.astype(float); ok=np.isfinite(y)
    if ok.sum()<5: return sr
    c=np.polyfit(x[ok],y[ok],1); return pd.Series(y-np.polyval(c,x),index=sr.index)

fig,ax=plt.subplots(figsize=(9,4)); res={}
for crop,mon,col in [('Wheat',6,'#d68910'),('Sunflower',8,'#229954')]:
    y=pd.read_excel('Yield.xls',sheet_name=crop,header=0)
    yl=y.melt(id_vars='Year',var_name='name',value_name='yield').rename(columns={'Year':'year'}); yl['key']=yl['name'].map(fold)
    sp=S[S.month==mon][['key','year']+[f'SPEI{k}' for k in SCALES]]
    m=yl.merge(sp,on=['key','year'],how='inner'); m['yld_dt']=np.nan
    for kk,g in m.groupby('key'):
        m.loc[g.index,'yld_dt']=detrend(g.set_index('year')['yield']).values
    rs=[]
    for k in SCALES:
        x,z=clean(m['yld_dt'].values,m[f'SPEI{k}'].values); rs.append(stats.pearsonr(x,z)[0])
    res[crop]=rs; ax.plot([str(k) for k in SCALES],rs,'o-',color=col,label=crop)
ax.axhline(0,color='k',lw=.5); ax.set_xlabel('SPEI scale (months)'); ax.set_ylabel('r (detrended yield)')
ax.set_title('Yield–SPEI correlation'); ax.legend(); plt.tight_layout(); plt.show()
print('SPEI scale:',SCALES)
for c,r in res.items(): print(f'  {c:10}', [round(v,3) for v in r])

## 5 · Robustness (α × season)

The headline uses α = 0.5 (independent of any drought index). Repeating the ranking under
α = 0.34 (regional mean) and three season windows confirms SPEI-3 is the best scale in almost
every scenario.

In [ ]:
alphas={'α=0.5':0.5,'α=0.34':0.34}; windows={'Jun-Aug':[6,7,8],'Jun-Oct':[6,7,8,9,10],'Apr-Oct':[4,5,6,7,8,9,10]}
print(f"{'scenario':22}"+''.join(f"{'SPEI'+str(k):>8}" for k in SCALES)+f"{'best':>9}")
for an,av in alphas.items():
    v=vhi.copy(); v['VHI']=av*v['VCI']+(1-av)*v['TCI']
    m=v.merge(S.drop(columns='name'),on=['key','year','month'],how='inner')
    for wn,mo in windows.items():
        dd=m[m.month.isin(mo)]; rr={}
        for k in SCALES:
            x,y=clean(dd['VHI'].values,dd[f'SPEI{k}'].values); rr[k]=stats.pearsonr(x,y)[0]
        bk=max(rr,key=rr.get)
        print(f"{an+' | '+wn:22}"+''.join(f"{rr[k]:>8.2f}" for k in SCALES)+f"{'SPEI-'+str(bk):>9}")

## Summary

- **SPEI-3 is the selected scale** — top composite (50/50), best Pearson/Spearman/MI, largest
  dose–response swing, lag-0 optimal. SPEI-6 is the close runner-up.
- **12-month scales fail** to discriminate dry vs healthy vegetation (composite ≤ 13).
- **Sunflower yield** validates SPEI-3/SPEI-6 (r ≈ 0.45–0.51); wheat is weak (winter buffered).
- **Robust** to α (0.5 vs 0.34) and season window.
- **İpsala** is a known outlier (irrigated valley) — consider excluding it in a sensitivity run.

Next: VHI raster maps, a Hargreaves-PET cross-check of SPEI, and (if extending the paper) a
mixed-effects panel model of yield on SPEI with district random effects.
